# Describe de la muestra estratificada

Consume los Parquet generados por [02_stratified_sampling.ipynb](02_stratified_sampling.ipynb) en `data/samples/` y corre un `describe()` sobre reviews y metadata.

In [1]:
from pathlib import Path

import pandas as pd

SAMPLES_DIR = Path("../data/samples")

reviews_df = pd.read_parquet(SAMPLES_DIR / "reviews_sample.parquet")
meta_df = pd.read_parquet(SAMPLES_DIR / "meta_sample.parquet")

print(f"reviews: {reviews_df.shape}")
print(f"metadata: {meta_df.shape}")

reviews: (30000, 11)
metadata: (20959, 15)


In [2]:
def split_list_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    """Separa columnas con listas/dicts (no hasheables) del resto.

    `describe(include="all")` cuelga sobre columnas tipo `images`/`features`/`details`, etc.,
    porque para calcular unique/top/freq compara elemento a elemento cuando el tipo no es
    hasheable. Se las excluye del describe estándar y se resumen aparte (null count, largo medio).
    """
    list_cols, plain_cols = [], []
    for col in df.columns:
        sample = df[col].dropna()
        is_list_like = len(sample) > 0 and isinstance(sample.iloc[0], (list, dict))
        (list_cols if is_list_like else plain_cols).append(col)
    return list_cols, plain_cols


def summarize_list_columns(df: pd.DataFrame, list_cols: list[str]) -> pd.DataFrame:
    def length(value):
        if value is None:
            return None
        try:
            return len(value)
        except TypeError:
            return None

    return pd.DataFrame(
        {
            "non_null": [df[c].notna().sum() for c in list_cols],
            "avg_len": [df[c].map(length).mean() for c in list_cols],
        },
        index=list_cols,
    )

## Reviews

In [3]:
reviews_list_cols, reviews_plain_cols = split_list_columns(reviews_df)
reviews_df[reviews_plain_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
rating,30000.0,3.000000e+00,1.414237e+00,1.000000e+00,2.000000e+00,3.000000e+00,4.000000e+00,5.000000e+00
timestamp,30000.0,1.522463e+12,9.154190e+10,9.440732e+11,1.460392e+12,1.528471e+12,1.596660e+12,1.693229e+12
helpful_vote,30000.0,1.998333e+00,1.972734e+01,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,2.657000e+03


In [4]:
summarize_list_columns(reviews_df, reviews_list_cols)

,non_null,avg_len


## Metadata

In [5]:
meta_list_cols, meta_plain_cols = split_list_columns(meta_df)
meta_df[meta_plain_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
average_rating,20959.0,3.716928,0.702447,1.0,3.4,3.80,4.20,5.00
rating_number,20809.0,1437.806045,19157.170960,1.0,9.0,37.00,187.00,1898759.00
price,6129.0,12.858367,29.077981,0.0,0.0,2.32,15.85,498.73


In [6]:
summarize_list_columns(meta_df, meta_list_cols)

,non_null,avg_len
details,20959,291.0
